# Bölüm 4: Pandas ile Veriyi Keşfetme

Detaylı sipariş verisini çekip Pandas ile analiz edeceğiz:
- `shape`, `info()`, `describe()` ile genel bakış
- `groupby()` ile gruplama
- Filtreleme ve çapraz analiz

In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="1234"
)

## Detaylı sipariş verisini çekelim

In [2]:
df_orders = pd.read_sql("""
    SELECT
        o.order_id,
        c.company_name AS musteri,
        c.country AS ulke,
        cat.category_name AS kategori,
        p.product_name AS urun,
        od.unit_price AS fiyat,
        od.quantity AS miktar,
        (od.unit_price * od.quantity) AS toplam,
        o.order_date AS tarih
    FROM orders o
    JOIN order_details od ON o.order_id = od.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON p.product_id = od.product_id
    JOIN categories cat ON cat.category_id = p.category_id
""", conn)

print(f"Toplam {df_orders.shape[0]} satir, {df_orders.shape[1]} kolon")
df_orders.head()

Toplam 2155 satir, 9 kolon


/var/folders/_7/4xlcmys94rvbr77bgf94cd180000gn/T/ipykernel_64600/4245507029.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_orders = pd.read_sql("""


,order_id,musteri,ulke,kategori,urun,fiyat,miktar,toplam,tarih
0,10248,Vins et alcools Chevalier,France,Dairy Products,Queso Cabrales,14.0,12,168.000000,1996-07-04
1,10248,Vins et alcools Chevalier,France,Grains/Cereals,Singaporean Hokkien Fried Mee,9.8,10,98.000002,1996-07-04
2,10248,Vins et alcools Chevalier,France,Dairy Products,Mozzarella di Giovanni,34.8,5,173.999996,1996-07-04
3,10249,Toms Spezialitäten,Germany,Produce,Tofu,18.6,9,167.400003,1996-07-05
4,10249,Toms Spezialitäten,Germany,Produce,Manjimup Dried Apples,42.4,40,1696.000061,1996-07-05


### Kullanılan fonksiyonlar:
- **`pd.read_sql(sorgu, baglanti)`** — SQL sorgusunu çalıştırıp sonucu DataFrame olarak döndürür
- **`.shape`** — (satır sayısı, kolon sayısı) şeklinde boyut bilgisi verir
- **`.head()`** — İlk 5 satırı gösterir (parametre vererek değiştirebilirsiniz: `.head(10)`)

## Genel bakış — info(), describe()

In [3]:
# info() — Her kolonun adını, tipini ve kaç tane dolu veri olduğunu gösterir
# Mesela "object" = metin, "float64" = ondalıklı sayı, "int64" = tam sayı
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 2155 entries, 0 to 2154
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  2155 non-null   int64  
 1   musteri   2155 non-null   str    
 2   ulke      2155 non-null   str    
 3   kategori  2155 non-null   str    
 4   urun      2155 non-null   str    
 5   fiyat     2155 non-null   float64
 6   miktar    2155 non-null   int64  
 7   toplam    2155 non-null   float64
 8   tarih     2155 non-null   object 
dtypes: float64(2), int64(2), object(1), str(4)
memory usage: 151.7+ KB


In [6]:
# describe() — Sadece sayısal kolonlar için istatistik özeti verir:
# count = kaç veri var, mean = ortalama, std = standart sapma,
# min/max = en küçük/büyük, 25%/50%/75% = yüzdelik dilimler
df_orders.describe().round(2)

,order_id,fiyat,miktar,toplam
count,2155.00,2155.00,2155.00,2155.00
mean,10659.38,26.22,23.81,628.52
std,241.38,29.83,19.02,1036.47
min,10248.00,2.00,1.00,4.80
25%,10451.00,12.00,10.00,154.00
50%,10657.00,18.40,20.00,360.00
75%,10862.50,32.00,30.00,722.25
max,11077.00,263.50,130.00,15810.00


In [7]:
# isnull().sum() — Her kolonda kaç tane eksik (boş) veri olduğunu sayar
# 0 ise o kolonda eksik veri yok demektir
df_orders.isnull().sum()

order_id    0
musteri     0
ulke        0
kategori    0
urun        0
fiyat       0
miktar      0
toplam      0
tarih       0
dtype: int64

## Ülkelere göre toplam satış
Sizce hangi ülke en çok satış yapıyordur?

In [10]:
# groupby('ulke') — Veriyi ülkelere göre gruplar
# ['toplam'].sum() — Her gruptaki "toplam" kolonunu toplar
# sort_values() — Sonucu büyükten küçüğe sıralar
ulke_satis = df_orders.groupby('ulke')['toplam'].sum().sort_values(ascending=False)
ulke_satis.head(10).round(2)

ulke
USA          263566.98
Germany      244640.63
Austria      139496.63
Brazil       114968.48
France        85498.76
Venezuela     60814.89
UK            60616.51
Sweden        59523.70
Ireland       57317.39
Canada        55334.10
Name: toplam, dtype: float64

## Kategorilere göre ortalama sipariş tutarı
En pahalı kategori hangisi?

In [11]:
# .mean() — sum() yerine ortalama hesaplar
# .round(2) — Sonucu 2 ondalık basamağa yuvarlar
kategori_ort = df_orders.groupby('kategori')['toplam'].mean().sort_values(ascending=False)
kategori_ort.round(2)

kategori
Meat/Poultry      1029.99
Produce            774.03
Beverages          709.23
Dairy Products     686.70
Confections        530.24
Condiments         526.36
Grains/Cereals     513.91
Seafood            429.16
Name: toplam, dtype: float64

## En çok satan 10 ürün (adet)

In [12]:
# Aynı mantık: ürüne göre grupla, miktarı topla, sırala
# .head(10) — Sadece ilk 10 sonucu göster
urun_satis = df_orders.groupby('urun')['miktar'].sum().sort_values(ascending=False)
urun_satis.head(10)

urun
Camembert Pierrot         1577
Raclette Courdavault      1496
Gorgonzola Telino         1397
Gnocchi di nonna Alice    1263
Pavlova                   1158
Rhönbräu Klosterbier      1155
Guaraná Fantástica        1125
Boston Crab Meat          1103
Tarte au sucre            1083
Chang                     1057
Name: miktar, dtype: int64

## Bonus: Çapraz analiz — Ülke bazında kategori karşılaştırma

In [13]:
# df[df['kolon'] == 'deger'] — Filtreleme: sadece koşulu sağlayan satırları alır
# SQL'deki WHERE gibi düşünün

usa = df_orders[df_orders['ulke'] == 'USA']
print("USA:")
print(usa.groupby('kategori')['toplam'].sum().sort_values(ascending=False).round(2))

print()

de = df_orders[df_orders['ulke'] == 'Germany']
print("Germany:")
print(de.groupby('kategori')['toplam'].sum().sort_values(ascending=False).round(2))

USA:
kategori
Beverages         63361.15
Meat/Poultry      45394.06
Dairy Products    41549.30
Confections       38804.05
Seafood           25025.37
Grains/Cereals    20411.30
Condiments        18555.85
Produce           10465.90
Name: toplam, dtype: float64

Germany:
kategori
Beverages         57644.60
Dairy Products    53170.90
Confections       37799.44
Seafood           24154.10
Meat/Poultry      22607.44
Condiments        17395.10
Produce           17265.90
Grains/Cereals    14603.15
Name: toplam, dtype: float64


## Pandas Fonksiyon Özeti

| Fonksiyon | Ne yapar? | SQL karşılığı |
|-----------|-----------|---------------|
| `df.head(n)` | İlk n satırı gösterir | `LIMIT n` |
| `df.shape` | (satır, kolon) sayısını verir | `SELECT COUNT(*)` |
| `df.info()` | Kolon tipleri ve boş veri bilgisi | — |
| `df.describe()` | Sayısal istatistikler (ort, min, max...) | — |
| `df.isnull().sum()` | Eksik veri sayısı | `WHERE ... IS NULL` |
| `df.groupby('x')['y'].sum()` | Gruplama + toplam | `GROUP BY x` + `SUM(y)` |
| `df.groupby('x')['y'].mean()` | Gruplama + ortalama | `GROUP BY x` + `AVG(y)` |
| `df.sort_values()` | Sıralama | `ORDER BY` |
| `df[df['x'] == 'a']` | Filtreleme | `WHERE x = 'a'` |
| `.round(2)` | Ondalık yuvarlama | `ROUND(..., 2)` |